In [ ]:
#----------Maximum flow and minimum cut 
# How many cancer patients can the hospital network transfer to treatment facilities, and which connections 
# restrict that capacity?
# Suppose each connection has a daily patient-transfer capacity.
"""
          Cancer Clinic
                V
    Route A - 3 patients/day
    Route B - 2 patients/day
                V
        Treatment Center

1) Maximum flow: The greatest number of patients that can be transferred per day under the stated capacities.
2) Minimum cut: The smallest total capacity of connections whose removal separates the source from the destination. 
It identifies a network bottleneck.

The network can transfer at most 5 patients per day under these assumptions. Increasing treatment-center capacity alone will 
not help if the outgoing referral capacity from the clinic remains unchanged.
Business value: Use this concept to identify capacity constraints in referrals, chemotherapy services, diagnostic facilities, 
or treatment-center networks. Real patient flow also needs to account for demand, staffing, clinical suitability, and scheduling.
"""
from collections import deque
capacity = {                                # Directed edges: (from, to): daily patient capacity
    ("Cancer_Clinic", "Route_A"): 3,        # Here key is a tuple containing two elements
    ("Cancer_Clinic", "Route_B"): 2,
    ("Route_A", "Route_B"): 1,
    ("Route_A", "Treatment_Center"): 2,
    ("Route_B", "Treatment_Center"): 3
}

def maximum_flow_and_minimum_cut(capacity, source, sink):
    residual = {}
    for (u, v), amount in capacity.items():               # In this for loop, we are using tuple to take care complex dic key of capacity
        residual[(u, v)] = amount
        residual[(v, u)] = 0
    flow = 0
    while True:
        parent = {source: None}                            # BFS finds a route with remaining capacity.
        queue = deque([source])
        while queue and sink not in parent:
            u = queue.popleft()
            for (a, b), remaining in residual.items():
                if a == u and remaining > 0 and b not in parent:
                    parent[b] = u
                    queue.append(b)
        if sink not in parent:
            break                                           #---- No more usable routes.
        bottleneck = float("inf")                           # Find the smallest capacity on this route.
        v = sink
        while v != source:
            u = parent[v]
            bottleneck = min(bottleneck, residual[(u, v)])
            v = u
        v = sink                                            # Update remaining capacities.
        while v != source:
            u = parent[v]
            residual[(u, v)] -= bottleneck
            residual[(v, u)] += bottleneck
            v = u
        flow += bottleneck
    reachable = {source}                                    # Find nodes still reachable from source.
    queue = deque([source])
    while queue:
        u = queue.popleft()
        for (a, b), remaining in residual.items():
            if a == u and remaining > 0 and b not in reachable:
                reachable.add(b)
                queue.append(b)
    min_cut = [  (u, v, amount) for (u, v), amount in capacity.items()         # Original edges crossing the final cut.
        if u in reachable and v not in reachable
    ]
    return flow, min_cut
    
flow, cut = maximum_flow_and_minimum_cut( capacity, "Cancer_Clinic", "Treatment_Center")
print("Maximum patient flow per day:", flow)
print("Minimum-cut connections:", cut)
print("Minimum-cut capacity:", sum(edge[2] for edge in cut))

Maximum patient flow per day: 5
Minimum-cut connections: [('Cancer_Clinic', 'Route_A', 3), ('Cancer_Clinic', 'Route_B', 2)]
Minimum-cut capacity: 5
